# Food.com Baseline Evaluation

Dieses Notebook führt unsere erste saubere Offline-Evaluation auf dem Food.com-Dataset aus. Wir behandeln das Problem als Top-N-Recommendation mit positivem implizitem Feedback.

Modellierungsannahme für diese erste Version:

- positives Feedback: `rating >= 4`
- nur User mit mindestens 2 positiven Interaktionen werden evaluiert
- Split: strict leave-last-out nach Zeitstempel

## 1. Imports And Setup

Wir laden die Evaluationsfunktionen und die Modelle, setzen Seeds fuer Reproduzierbarkeit und definieren zentrale Laufzeit-Schalter. `RUN_CONTENT_BASED` ist standardmaessig aktiv, `RUN_KNN_ON_FULL_DATA` bleibt aus Performance-Gruenden deaktiviert.

In [1]:
import os
import sys
import csv
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "evaluation").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Current working directory:", cwd)
print("Project root used:", project_root)

from evaluation.split import leave_last_out_split
from evaluation.metrics import recall_at_k, ndcg_at_k, catalog_coverage_at_k, novelty_at_k
from models.baselines import PopularityRecommender, RandomRecommender, TrendingRecommender
from models.content_based import ContentBasedRecommender, HybridMFContentRecommender
from models.knn import ItemItemKNN
from models.mf import BiasedMatrixFactorization, BPRMatrixFactorization

np.random.seed(42)
random.seed(42)
K = 20
RUN_KNN_ON_FULL_DATA = False
RUN_CONTENT_BASED = True
CONTENT_CANDIDATE_ITEMS = 20000


Current working directory: /Users/isabellenachbaur/Documents/FHGR/Recommender Systems/projekt/recommended-systems/notebooks
Project root used: /Users/isabellenachbaur/Documents/FHGR/Recommender Systems/projekt/recommended-systems


## 2. Data Pipeline

Hier bauen wir den finalen Modellierungsdatensatz auf:

- Food.com-RAW-Dateien laden
- Ratings in positives implizites Feedback umwandeln
- nur evaluierbare User behalten
- Rezept-Metadaten joinen
- leave-last-out Split vorbereiten

In [2]:
print("=== PHASE 1: DATA PIPELINE ===")

interactions = pd.read_csv("../data/raw/RAW_interactions.csv")
recipes = pd.read_csv("../data/raw/RAW_recipes.csv")

interactions["date"] = pd.to_datetime(interactions["date"], errors="coerce")
recipes["submitted"] = pd.to_datetime(recipes["submitted"], errors="coerce")

# Positive implicit feedback: ratings 4 and 5.
positive_df = interactions[interactions["rating"] >= 4].copy()
positive_df = positive_df.rename(columns={"date": "date_time"})
positive_df = positive_df.sort_values(["user_id", "date_time"])

# Leave-last-out needs at least two positive interactions per user.
user_counts = positive_df["user_id"].value_counts()
valid_users = user_counts[user_counts >= 2].index
positive_df = positive_df[positive_df["user_id"].isin(valid_users)].copy()

recipes_model = recipes.rename(columns={"id": "recipe_id"}).copy()
df_model = positive_df.merge(recipes_model, on="recipe_id", how="inner")

print(f"Positive interactions after filtering: {len(df_model):,}")
print(f"Users for evaluation: {df_model['user_id'].nunique():,}")
print(f"Recipes in final dataset: {df_model['recipe_id'].nunique():,}")
print(f"Date range: {df_model['date_time'].min()} to {df_model['date_time'].max()}")

train_df, test_df = leave_last_out_split(df_model, user_col="user_id", time_col="date_time")

train_item_counts = train_df["recipe_id"].value_counts().to_dict()
total_train_interactions = len(train_df)
total_items = train_df["recipe_id"].nunique()
recipe_names = (
    recipes_model.drop_duplicates("recipe_id")
    .assign(name=lambda x: x["name"].fillna("Unknown Recipe"))
    .set_index("recipe_id")["name"]
    .to_dict()
)

print("\nPreview of final training data:")
display(train_df[["user_id", "recipe_id", "date_time", "rating", "name"]].head(5))

=== PHASE 1: DATA PIPELINE ===
Positive interactions after filtering: 875,778
Users for evaluation: 52,364
Recipes in final dataset: 206,817
Date range: 2000-01-25 00:00:00 to 2018-12-19 00:00:00
Sorting data chronologically to prevent time leakage...
Dropping users with tied final timestamps because a strict leave-last-out split is not identifiable: 8115 users
Splitting data: Leave-last-out...
Train Set: 707232 interactions
Test Set: 44249 interactions

Preview of final training data:


,user_id,recipe_id,date_time,rating,name
0,1533,17338,2002-02-19,4,zucchini and rice casserole
1,1533,24375,2002-04-23,5,costa rican stuffed tortilla
2,1533,10721,2002-05-02,5,orange yogurt cream
3,1533,23891,2002-05-06,5,parmesan fish in the oven
4,1533,24136,2002-05-20,5,fennel mashed potatoes


## 3. Model Training

Wir trainieren die Collaborative-Filtering-Baselines und zusaetzlich einen Content-Based-Recommender. Fuer Content-Based begrenzen wir die Kandidatenmenge auf die populaersten Trainingsrezepte, damit TF-IDF und Cosine Scoring auf dem grossen Food.com-Katalog praktikabel bleiben.

In [ ]:
print("\n=== PHASE 2: MODEL TRAINING ===")

pop_model = PopularityRecommender()
pop_model.fit(train_df, item_col="recipe_id")

trend_model = TrendingRecommender(days_window=180)
trend_model.fit(train_df, item_col="recipe_id", time_col="date_time")

rand_model = RandomRecommender()
rand_model.fit(train_df, item_col="recipe_id")

content_model = None
if RUN_CONTENT_BASED:
    content_candidate_ids = train_df["recipe_id"].value_counts().head(CONTENT_CANDIDATE_ITEMS).index
    content_items_df = recipes_model[recipes_model["recipe_id"].isin(content_candidate_ids)].copy()
    content_train_df = train_df[train_df["recipe_id"].isin(content_candidate_ids)].copy()

    content_model = ContentBasedRecommender(
        text_cols=["name", "tags", "ingredients", "description"],
        metadata_cols=["minutes", "n_steps", "n_ingredients"],
        min_df=2,
        ngram_range=(1, 1),
        time_col="date_time",
        recency_decay=0.0,
    )
    print(f"Training Content-Based model on top {len(content_candidate_ids):,} recipes by train popularity...")
    content_model.fit(
        content_train_df,
        content_items_df,
        user_col="user_id",
        item_col="recipe_id"
    )

knn_model = None
if RUN_KNN_ON_FULL_DATA:
    knn_model = ItemItemKNN(k_neighbors=100, shrinkage=10)
    knn_model.fit(train_df, item_col="recipe_id")
else:
    print("Skipping Item-Item kNN on the full Food.com dataset because the current implementation is too expensive at this item scale.")

mf_model = BiasedMatrixFactorization(k_factors=16, epochs=5)
mf_model.fit(train_df, item_col="recipe_id")

bpr_model = BPRMatrixFactorization(k_factors=32, epochs=5)
bpr_model.fit(train_df, item_col="recipe_id")

hybrid_model = None
if content_model is not None:
    hybrid_model = HybridMFContentRecommender(
        mf_model=bpr_model,
        content_model=content_model,
        alpha=0.9,
        adaptive=True,
        sparse_threshold=5
    )

print("\nAll enabled models trained and ready for evaluation.")



=== PHASE 2: MODEL TRAINING ===
Training Popularity Baseline...
Learned popularity ranking for 185226 unique recipes.
Training Trending Baseline (last 180 days)...
Learned 795 trending recipes.
Training Random Baseline...
Random candidate universe contains 185226 recipes.
Training Content-Based model on top 20,000 recipes by train popularity...
Training Content-Based Recommender...
Using text columns: ['name', 'tags', 'ingredients', 'description']
Using metadata columns: ['minutes', 'n_steps', 'n_ingredients']
Feature dimension: 15691
Vocabulary size: 15691
Missing rates by feature: {'name': 0.0, 'tags': 0.0, 'ingredients': 0.0, 'description': 0.02335, 'minutes': 0.0, 'n_steps': 0.0, 'n_ingredients': 0.0}
Content model trained with 20000 items and 35806 user profiles.
Skipping Item-Item kNN on the full Food.com dataset because the current implementation is too expensive at this item scale.
Training Biased MF (k=16, epochs=5)...
Epoch 1/5 completed.
Epoch 2/5 completed.
Epoch 3/5 compl

KeyboardInterrupt: 

## 4. Offline Evaluation

Jetzt evaluieren wir alle aktivierten Modelle auf demselben leave-last-out Split mit Recall@20, NDCG@20, Coverage@20 und Novelty@20. Content-Based wird dabei als eigene Baseline ausgewiesen.

In [ ]:
print("\n=== PHASE 3: MASTER EVALUATION LOOP ===")

results = {
    "Popularity": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Trending (180d)": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Random": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Content-Based": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Biased MF (SQ)": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "BPR MF": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []},
    "Hybrid BPR+Content": {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []}
}

models = {
    "Popularity": pop_model,
    "Trending (180d)": trend_model,
    "Random": rand_model,
    "Content-Based": content_model,
    "Biased MF (SQ)": mf_model,
    "BPR MF": bpr_model,
    "Hybrid BPR+Content": hybrid_model
}

if hybrid_model is None:
    results.pop("Hybrid BPR+Content")
    models.pop("Hybrid BPR+Content")

if content_model is None:
    results.pop("Content-Based")
    models.pop("Content-Based")

if knn_model is not None:
    results["Item-Item kNN"] = {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []}
    models["Item-Item kNN"] = knn_model

print("Preparing user histories...")
train_history = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

test_users = test_df["user_id"].unique()
print(f"Evaluating {len(test_users):,} users. This can take a while...")

for user in test_users:
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
    user_hist = train_history.get(user, set())

    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)
        results[name]["all_recs"].append(recs)
        results[name]["recalls"].append(recall_at_k(recs, [true_item], k=K))
        results[name]["ndcgs"].append(ndcg_at_k(recs, [true_item], k=K))
        results[name]["novelties"].append(
            novelty_at_k(recs, train_item_counts, total_train_interactions, k=K)
        )

print("\n--- FINAL MACRO-AVERAGED RESULTS TABLE ---")
print(f"{'Model':<20} | {'Recall@20':<10} | {'NDCG@20':<10} | {'Coverage@20':<12} | {'Novelty@20'}")
print("-" * 75)

for name in models.keys():
    cov = catalog_coverage_at_k(results[name]["all_recs"], total_items)
    mean_rec = np.mean(results[name]["recalls"])
    mean_ndcg = np.mean(results[name]["ndcgs"])
    mean_nov = np.mean(results[name]["novelties"])
    print(f"{name:<20} | {mean_rec:.4f}     | {mean_ndcg:.4f}     | {cov:.4%} | {mean_nov:.4f}")
    
if np.mean(results["Popularity"]["recalls"]) > 0.15:
    print("\nWARNING: Popularity baseline is very strong (>15% Recall). Check whether the split is too easy or too popular-item-heavy.")
else:
    print("\nSanity checks passed.")



=== PHASE 3: MASTER EVALUATION LOOP ===
Preparing user histories...
Evaluating 44,249 users. This can take a while...

--- FINAL MACRO-AVERAGED RESULTS TABLE ---
Model            | Recall@20  | NDCG@20    | Coverage@20  | Novelty@20
---------------------------------------------------------------------------
Popularity       | 0.0312     | 0.0125     | 0.0367% | 10.3203
Trending (180d)  | 0.0029     | 0.0018     | 0.0173% | 15.6854
Random           | 0.0001     | 0.0000     | 99.1702% | 18.3655
Content-Based    | 0.0083     | 0.0035     | 9.9376% | 14.4408
Biased MF (SQ)   | 0.0264     | 0.0090     | 0.1091% | 11.1796
BPR MF           | 0.0289     | 0.0105     | 0.2473% | 10.5064

Sanity checks passed.


In [ ]:
print("\n--- TWO-STAGE RECOMMENDER INTERPRETATION ---")

print("""
Stage 1: Candidate Generation (Retriever)
------------------------------------------
BPR MF:
- behavioral retrieval using collaborative filtering
- retrieves items from latent user-item interactions

Content-Based:
- semantic retrieval using recipe metadata
- retrieves items using ingredients, tags, and descriptions

Popularity:
- fallback candidate generator
- strong for sparse users and cold-start scenarios

Stage 2: Ranking
----------------
Hybrid BPR + Content:
- merges collaborative and content-based candidates
- reranks retrieved items using reciprocal-rank blending

Interpretation:
---------------
The ranker can only reorder what retrieval keeps alive.
If retrieval misses a relevant recipe, ranking cannot recover it.

This follows the standard two-stage recommender design:
cheap high-recall retrieval first, precise ranking second.
""")

In [ ]:
print("\n--- USER ACTIVITY SLICE ANALYSIS ---")

user_train_counts = train_df.groupby("user_id")["recipe_id"].count().to_dict()

def user_activity_group(user_id):
    n = user_train_counts.get(user_id, 0)
    if n <= 3:
        return "Sparse users (<=3)"
    elif n <= 10:
        return "Medium users (4-10)"
    else:
        return "Heavy users (>10)"

slice_rows = []

for user in test_users:
    group = user_activity_group(user)
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
    user_hist = train_history.get(user, set())

    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)
        slice_rows.append({
            "model": name,
            "slice": group,
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K),
        })

slice_df = pd.DataFrame(slice_rows)

slice_summary = (
    slice_df
    .groupby(["model", "slice"])
    .agg(
        users=("recall", "count"),
        recall_at_20=("recall", "mean"),
        ndcg_at_20=("ndcg", "mean")
    )
    .reset_index()
)

display(slice_summary)

In [ ]:
print("\n--- COLD ITEM SLICE ANALYSIS ---")

cold_items = ContentBasedRecommender.cold_item_set(
    train_df,
    test_df,
    item_col="recipe_id"
)

print(f"Cold items in test set: {len(cold_items)}")

cold_rows = []

for user in test_users:
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]

    # only evaluate users whose test item is a cold item
    if true_item not in cold_items:
        continue

    user_hist = train_history.get(user, set())

    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)

        cold_rows.append({
            "model": name,
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K),
        })

cold_df = pd.DataFrame(cold_rows)

if len(cold_df) == 0:
    print("No cold items found in test set under current split.")
else:
    cold_summary = (
        cold_df
        .groupby("model")
        .agg(
            users=("recall", "count"),
            recall_at_20=("recall", "mean"),
            ndcg_at_20=("ndcg", "mean")
        )
        .reset_index()
    )

    display(cold_summary)

In [ ]:
print("\n--- HYBRID FINAL ALPHA CHECK ---")

'''
sample_users = test_users[:1000]

Tested different alpha values for the Hybrid BPR+Content model 
- to see how the balance between collaborative and content-based retrieval affects performance. 

alpha_values = [0.3, 0.5, 0.7, 0.9]
alpha_rows = []

for alpha in alpha_values:
    temp_hybrid = HybridMFContentRecommender(
        mf_model=bpr_model,
        content_model=content_model,
        alpha=alpha,
        adaptive=True,
        sparse_threshold=5
    )

    recalls = []
    ndcgs = []
    all_recs = []

    for user in sample_users:
        true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
        user_hist = train_history.get(user, set())

        recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)

        recalls.append(recall_at_k(recs, [true_item], k=K))
        ndcgs.append(ndcg_at_k(recs, [true_item], k=K))
        all_recs.append(recs)

    alpha_rows.append({
        "alpha": alpha,
        "recall_at_20": np.mean(recalls),
        "ndcg_at_20": np.mean(ndcgs),
        "coverage_at_20": catalog_coverage_at_k(all_recs, total_items)
    })

alpha_df = pd.DataFrame(alpha_rows)
display(alpha_df)
'''


--- HYBRID FINAL ALPHA CHECK ---


'\nsample_users = test_users[:1000]\n\nTested different alpha values for the Hybrid BPR+Content model \n- to see how the balance between collaborative and content-based retrieval affects performance. \nalpha_values = [0.3, 0.5, 0.7, 0.9]\n\nResult summary: \nalpha = 0.9 achieved the best Recall@20 and NDCG@20 while still keeping strong catalog coverage.\n\nInterpretation:\nFood.com benefits more from stronger collaborative filtering (BPR) than from stronger content weighting.\n\nalpha_values = [0.9]\nalpha_rows = []\n\nfor alpha in alpha_values:\n    temp_hybrid = HybridMFContentRecommender(\n        mf_model=bpr_model,\n        content_model=content_model,\n        alpha=alpha,\n        adaptive=True,\n        sparse_threshold=5\n    )\n\n    recalls = []\n    ndcgs = []\n    all_recs = []\n\n    for user in sample_users:\n        true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]\n        user_hist = train_history.get(user, set())\n\n        recs = temp_hybrid.recom

In [ ]:
print("\n--- HYBRID FINAL ---")

sample_users = test_users[:1000]

# tested: [0.3, 0.5, 0.7, 0.9]--> best result: alpha = 0.9
# reason: best Recall@20 + NDCG@20 with strong coverage
# interpretation: Food.com benefits more from stronger collaborative filtering (BPR) than from stronger content weighting.

# Therefore, the final hybrid model uses:

alpha = 0.9

temp_hybrid = HybridMFContentRecommender(
    mf_model=bpr_model,
    content_model=content_model,
    alpha=alpha,
    adaptive=True,
    sparse_threshold=5
)

recalls = []
ndcgs = []
all_recs = []

for user in sample_users:
    true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
    user_hist = train_history.get(user, set())

    recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)

    recalls.append(recall_at_k(recs, [true_item], k=K))
    ndcgs.append(ndcg_at_k(recs, [true_item], k=K))
    all_recs.append(recs)

alpha_df = pd.DataFrame([{
    "alpha": alpha,
    "recall_at_20": np.mean(recalls),
    "ndcg_at_20": np.mean(ndcgs),
    "coverage_at_20": catalog_coverage_at_k(all_recs, total_items)
}])

display(alpha_df)

## 5. Diagnostics And Explainability

Neben den Metriken schauen wir auf kurze Diagnostics: Content-Based-Erklaerungen, optional kNN-Erklaerungen, BPR-Bias und BPR-Latent-Space-Nachbarn.

In [ ]:
print("\n=== PHASE 4: DIAGNOSTICS & EXPLAINABILITY ===")

if content_model is not None:
    print("\n--- 4A. CONTENT-BASED EXPLAINABILITY ---")
    cb_users = [u for u in train_df["user_id"].value_counts().index if u in content_model.user_profiles]
    sample_user = cb_users[0]
    explained_cb_recs = content_model.recommend(
        sample_user,
        user_history=train_history.get(sample_user, set()),
        k=3,
        return_explanations=True
    )
    print(f"Generating content-based explanations for User {sample_user}:")
    for i, (rec_id, explanation) in enumerate(explained_cb_recs, start=1):
        recipe_name = recipe_names.get(rec_id, "Unknown Recipe")
        print(f"{i}. {recipe_name}\n   -> {explanation}")
else:
    print("\n--- 4A. CONTENT-BASED EXPLAINABILITY ---")
    print("Skipped because Content-Based was disabled.")

if knn_model is not None:
    print("\n--- 4B. kNN CONSTRUCTIVE EXPLAINABILITY ---")
    heavy_users = train_df["user_id"].value_counts()
    sample_user = heavy_users[heavy_users > 5].sample(1, random_state=42).index[0]
    print(f"Generating explanations for User {sample_user}:")
    explained_recs = knn_model.recommend(
        sample_user,
        user_history=train_history.get(sample_user, set()),
        k=3,
        return_explanations=True
    )
    for i, (rec_id, explanation) in enumerate(explained_recs, start=1):
        recipe_name = recipe_names.get(rec_id, "Unknown Recipe")
        print(f"{i}. {recipe_name}\n   -> {explanation}")
else:
    print("\n--- 4B. kNN CONSTRUCTIVE EXPLAINABILITY ---")
    print("Skipped because Item-Item kNN was disabled for the full Food.com run.")

print("\n--- 4C. BPR BIAS INSPECTION ---")
top_bias_indices = np.argsort(bpr_model.b_i)[-5:][::-1]
print("Top 5 Recipes by Learned Bias:")
for idx in top_bias_indices:
    raw_id = bpr_model.reverse_item_mapping[idx]
    print(f"   -> {recipe_names.get(raw_id, 'Unknown Recipe')} (Bias: {bpr_model.b_i[idx]:.4f})")

print("\n--- 4D. BPR LATENT SPACE INSPECTION ---")
anchor_id = train_df["recipe_id"].value_counts().index[0]
anchor_idx = bpr_model.item_mapping[anchor_id]
anchor_vector = bpr_model.Q[anchor_idx].reshape(1, -1)

similarities = cosine_similarity(anchor_vector, bpr_model.Q)[0]
nearest_indices = [idx for idx in np.argsort(similarities)[::-1] if idx != anchor_idx][:5]

print(f"Anchor recipe: {recipe_names.get(anchor_id, 'Unknown Recipe')}")
print("Nearest latent neighbors:")
for idx in nearest_indices:
    raw_id = bpr_model.reverse_item_mapping[idx]
    print(f"   -> {recipe_names.get(raw_id, 'Unknown Recipe')} (Latent Sim: {similarities[idx]:.4f})")



=== PHASE 4: DIAGNOSTICS & EXPLAINABILITY ===

--- 4A. CONTENT-BASED EXPLAINABILITY ---
Generating content-based explanations for User 37449:
1. garlic potatoes
   -> Recommended because it overlaps on: tags_low, tags_or, tags_fat.
2. roasted green beans with garlic and onions
   -> Recommended because it overlaps on: tags_low, tags_fat, tags_healthy.
3. simple cucumbers
   -> Recommended because it overlaps on: tags_low, tags_or, tags_fat.

--- 4B. kNN CONSTRUCTIVE EXPLAINABILITY ---
Skipped because Item-Item kNN was disabled for the full Food.com run.

--- 4C. BPR BIAS INSPECTION ---
Top 5 Recipes by Learned Bias:
   -> kittencal s 5 minute cinnamon flop brunch cake (Bias: 3.4203)
   -> to die for crock pot roast (Bias: 3.4195)
   -> creamy cajun chicken pasta (Bias: 3.4079)
   -> whatever floats your boat  brownies (Bias: 3.4008)
   -> kittencal s italian melt in your mouth meatballs (Bias: 3.3955)

--- 4D. BPR LATENT SPACE INSPECTION ---
Anchor recipe: to die for crock pot roast
N

## 6. Experiment Logging

Zum Schluss schreiben wir alle aktivierten Modelllaeufe in `runs/runs.csv`, damit wir spaeter verschiedene Varianten vergleichen koennen.

In [ ]:
print("\n=== PHASE 5: EXPERIMENT LOGGING ===")

runs_file = "../runs/runs.csv"
os.makedirs("../runs", exist_ok=True)

run_rows = []
for name, model in models.items():
    coverage = catalog_coverage_at_k(results[name]["all_recs"], total_items)
    hyperparams = ""
    if name == "BPR MF":
        hyperparams = f"k_factors={bpr_model.k}, epochs={bpr_model.epochs}, reg={bpr_model.reg}"
    elif name == "Biased MF (SQ)":
        hyperparams = f"k_factors={mf_model.k}, epochs={mf_model.epochs}, reg={mf_model.reg}"
    elif name == "Content-Based":
        hyperparams = f"top_items={CONTENT_CANDIDATE_ITEMS}, text=name/tags/ingredients/description, min_df={content_model.min_df}"
    elif name == "Trending (180d)":
        hyperparams = f"days_window={trend_model.days_window}"

    run_rows.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "dataset": "Food.com_RAW_rating_ge_4_strict_split",
        "model": name,
        "k_evaluated": K,
        "hyperparams": hyperparams,
        "recall": round(np.mean(results[name]["recalls"]), 4),
        "ndcg": round(np.mean(results[name]["ndcgs"]), 4),
        "coverage": round(coverage, 4)
    })

file_exists = os.path.isfile(runs_file)
with open(runs_file, mode="a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=run_rows[0].keys())
    if not file_exists:
        writer.writeheader()
    writer.writerows(run_rows)

print(f"Logged {len(run_rows)} model runs to {runs_file}!")
print("Notebook complete. Ready for comparison with future variants.")



=== PHASE 5: EXPERIMENT LOGGING ===
Logged 6 model runs to ../runs/runs.csv!
Notebook complete. Ready for comparison with future variants.
